In [1]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
from pdf2image import convert_from_path
from tqdm.notebook import tqdm
import shutil

In [2]:
#Actual code for manifest
poppler_path = "/Users/jimenagonzalez/miniforge3/envs/main/bin"
path_origin = '/Users/jimenagonzalez/research/DSPL/Hybrid_SL_Modeling/Traditional_Modeling/OSG/' #Path with the modeling output from OSG

#To save the Zooniverse files
path_data = '/Users/jimenagonzalez/research/DSPL/Hybrid_SL_Modeling/Zooniverse/Data/Citizens/' #Two folders here (model and mask)

bands = ['g', 'r', 'i']
folders = [f for f in os.listdir(path_origin) if os.path.isdir(os.path.join(path_origin, f))]
rows_model, rows_mask = [], []

files = os.listdir(path_origin) 
for file in files:
    if not file.endswith('_completeMasks.pdf'):
        continue
        
    coadd_id = int(file.split('_')[0])
    #print(coadd_id)
        
    for band in bands:
        name_image = f'{coadd_id}_model_{band}.png'
        rows_model.append({'COADD_OBJECT_ID': coadd_id, 'image_task': name_image})
        # Copy PNG to correct folder
        path1 = f'{path_origin}/{name_image}'
        path2 = f'{path_data}/Model/{name_image}'
        shutil.copy(path1, path2)
            
        
    name_image1 = f'{coadd_id}_pngNormal.png'
    name_image2 = f'{coadd_id}_mask.png'
    name_image3 = f'{coadd_id}_pngBlue.png'
    rows_mask.append({'COADD_OBJECT_ID': coadd_id, 'image_task1': name_image1, 'image_task2': name_image2, 'image_task3': name_image3})
    
    # Copy PNGs to correct folder
    path1 = f'{path_origin}/{name_image1}'
    path2 = f'{path_data}/Mask/{name_image1}'
    shutil.copy(path1, path2)

    path1 = f'{path_origin}/{name_image2}'
    path2 = f'{path_data}/Mask/{name_image2}'
    shutil.copy(path1, path2)

    path1 = f'{path_origin}/{name_image3}'
    path2 = f'{path_data}/Mask/{name_image3}'
    shutil.copy(path1, path2)
        

data_model = pd.DataFrame(rows_model)
data_mask = pd.DataFrame(rows_mask)

filename = '/Users/jimenagonzalez/research/DSPL/Hybrid_SL_Modeling/data/candidates_data/' + 'mycand_old.csv'
data_cand = pd.read_csv(filename)

data_model = data_model.merge(data_cand[['COADD_OBJECT_ID', 'EXP_SCORE']], on='COADD_OBJECT_ID', how='inner', validate='m:1',)
data_mask = data_mask.merge(data_cand[['COADD_OBJECT_ID', 'EXP_SCORE']], on='COADD_OBJECT_ID', how='inner', validate='m:1',)

#TODO: Rename columns to hide them from volunteers

#Separating by confidence category
data_modelA = data_model[data_model['EXP_SCORE'] >= 2.25]
data_modelB = data_model[(data_model['EXP_SCORE'] >= 1.25) & (data_model['EXP_SCORE'] < 2.25)] 

data_maskA = data_mask[data_mask['EXP_SCORE'] >= 2.25]
data_maskB = data_mask[(data_mask['EXP_SCORE'] >= 1.25) & (data_mask['EXP_SCORE'] < 2.25)]

data_modelA.to_csv(path_data + 'Model/manifest_modelA.csv', index=False)
data_modelB.to_csv(path_data + 'Model/manifest_modelB.csv', index=False)

data_maskA.to_csv(path_data + 'Mask/manifest_maskA.csv', index=False)
data_maskB.to_csv(path_data + 'Mask/manifest_maskB.csv', index=False)

In [3]:
data_model.head()

,COADD_OBJECT_ID,image_task,EXP_SCORE
0,1215104940,1215104940_model_g.png,1.275042
1,1215104940,1215104940_model_r.png,1.275042
2,1215104940,1215104940_model_i.png,1.275042
3,1305035224,1305035224_model_g.png,2.560902
4,1305035224,1305035224_model_r.png,2.560902
